# Intro backprop ninja

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt 
%matplotlib inline

In [2]:
# read in all the words
words = open('names.txt', 'r').read().splitlines()
print(words[:8])
print(max(len(w) for w in words))
print(len(words))

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']
15
32033


In [3]:
# build the vocab of chars and mapping to/from integers
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
vocab_size = len(itos)
print(itos)
print(vocab_size)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}
27


In [4]:
# build the dataset
block_size = 3     # context lenght: how many chars do we take to predict the next one?

def build_dataset(words):
    X, Y = [], []
    for w in words:
        
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]      # crop and append
            
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, Y.shape)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

torch.Size([182625, 3]) torch.Size([182625])
torch.Size([22655, 3]) torch.Size([22655])
torch.Size([22866, 3]) torch.Size([22866])


# let's get into action

In [5]:
def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | maxdiff: {maxdiff}')

In [6]:
n_emb = 10 # the dim of the character embedding vectors
n_hidden = 64 # the num of neurons in the hidden layer of the mlp

g = torch.Generator().manual_seed(2147483647)              # for reproducibility
C = torch.randn((vocab_size, n_emb),                generator=g) 

# Layer 1
W1 = torch.randn((n_emb * block_size, n_hidden),    generator=g) * (5/3)/((n_emb * block_size)**0.5)
b1 = torch.randn(n_hidden,                          generator=g) * 0.1  # using b1 just for fun 

# Layer 2
W2 = torch.randn((n_hidden, vocab_size),            generator=g) * 0.1
b2 = torch.randn(vocab_size,                        generator=g) * 0.1

# Batchnorm param
bngain = torch.rand((1, n_hidden)) * 0.1 + 1.0
bnbias = torch.rand((1, n_hidden)) * 0.1

parametres = [C, W1, b1, W2, b2, bngain, bnbias]
print(sum(p.nelement() for p in parametres)) # number of params in total
for p in parametres:
    p.requires_grad = True


4137


In [7]:
batch_size = 32
n = batch_size # shorter variable for convenience

#construct a mini batch
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix] # batch X, Y

# Forward pass

In [8]:
# forward pass, "chunckated" into smaller steps that are possible to backward one at a time

emb = C[Xb] # embed the characters into vectors
embcat = emb.view(emb.shape[0], -1) # concatenate the vectors

# Linear Layer 1
hprebn = embcat @ W1 + b1 #hidden pre-layer activation

#Batchnorm layer
bnmeani = 1/n*hprebn.sum(0, keepdim=True)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1 / (n-1)*(bndiff2).sum(0, keepdim=True)
bnvar_inv = (bnvar + 1e-5)**(-0.5)
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias

# Non linearity
h = torch.tanh(hpreact) # hidden layer

# Linear layer 2
logits = h @ W2 + b2 # output layer

# cross entropy loss (same as F.cross_entropy(logits, Yb))
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # substract max for numeral stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum**-1 # backprop wouldn't be exact with (1 / counts_sum)
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(n), Yb].mean()

# Pytorch backward pass
for p in parametres:
    p.grad = None
for t in [logprobs, probs, counts, counts_sum, counts_sum_inv, norm_logits, logit_maxes, logits, h, hpreact, bnraw, bnvar_inv, bnvar, bndiff2, bndiff, hprebn, bnmeani, embcat, emb]:
    t.retain_grad()
loss.backward()

loss


tensor(3.3482, grad_fn=<NegBackward0>)

# Backward pass

In [9]:
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = - 1 / n

dprobs = (1.0 / probs) * dlogprobs

dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)

dcounts_sum = (-1 * counts_sum **(-2)) * dcounts_sum_inv

dcounts = (counts_sum_inv * dprobs) + dcounts_sum

dnorm_logits = (norm_logits.exp()) * dcounts

dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True)

dlogits = dnorm_logits.clone() + F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes

dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)

dhpreact = (1.0 - h**2) * dh

dbngain = (bnraw * dhpreact).sum(0, keepdim=True)

dbnraw = dhpreact * bngain
dbnbias = dhpreact.sum(0, keepdim=True)
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)
dbnvar = (-0.5*(bnvar + 1e-5)**-1.5) * dbnvar_inv
dbndiff2 = (1.0 /(n-1))*torch.ones_like(bndiff2) * dbnvar
dbndiff = bnvar_inv * dbnraw + 2*bndiff*dbndiff2

dbnmeani = (-dbndiff).sum(0)
dhprebn = dbndiff.clone() + (1/n*torch.ones_like(hprebn)*dbnmeani)

dembcat = dhprebn @ W1.T 
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)
demb = dembcat.view(emb.shape)
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix = Xb[k, j]
        dC[ix] += demb[k, j]

In [10]:
cmp('logprobs', dlogprobs, logprobs)
cmp('probs', dprobs, probs)
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)
cmp('counts_sum', dcounts_sum, counts_sum)
cmp('counts', dcounts, counts)
cmp('norm_logits', dnorm_logits, norm_logits)
cmp('logit_maxes', dlogit_maxes, logit_maxes)
cmp('logits', dlogits, logits)
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)
cmp('hpreact', dhpreact, hpreact)
cmp('bngain', dbngain, bngain)
cmp('bnraw', dbnraw, bnraw)
cmp('bnbias', dbnbias, bnbias)

cmp('hbnvar', dbnvar, bnvar)
cmp('bndiff2', dbndiff2, bndiff2)
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)
cmp('bndiff', dbndiff, bndiff)
cmp('hprebn', dhprebn, hprebn)
cmp('bnmeani', dbnmeani, bnmeani)
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)
cmp('emb', demb, emb)
cmp('C', dC, C)

logprobs        | exact: True  | maxdiff: 0.0
probs           | exact: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | maxdiff: 0.0
counts_sum      | exact: True  | maxdiff: 0.0
counts          | exact: True  | maxdiff: 0.0
norm_logits     | exact: True  | maxdiff: 0.0
logit_maxes     | exact: True  | maxdiff: 0.0
logits          | exact: True  | maxdiff: 0.0
h               | exact: True  | maxdiff: 0.0
W2              | exact: True  | maxdiff: 0.0
b2              | exact: True  | maxdiff: 0.0
hpreact         | exact: False | maxdiff: 4.656612873077393e-10
bngain          | exact: False | maxdiff: 1.862645149230957e-09
bnraw           | exact: False | maxdiff: 6.984919309616089e-10
bnbias          | exact: False | maxdiff: 2.7939677238464355e-09
hbnvar          | exact: False | maxdiff: 9.313225746154785e-10
bndiff2         | exact: False | maxdiff: 2.9103830456733704e-11
bnvar_inv       | exact: False | maxdiff: 3.725290298461914e-09
bndiff          | exact: False | maxdiff: 4.

In [11]:
logprobs.shape

torch.Size([32, 27])

# Exercice 2

In [12]:
loss_fast = F.cross_entropy(logits, Yb)
print(loss_fast.item(), 'diff:', (loss_fast - loss).item())

3.348236083984375 diff: 0.0


In [13]:
# backward

dlogits = F.softmax(logits, 1)
dlogits[range(n), Yb] -= 1
dlogits /= n

cmp('logits', dlogits, logits)

logits          | exact: False | maxdiff: 6.05359673500061e-09


# Exercice 3

In [14]:
hpreact_fast = bngain * (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5) + bnbias
print('max diff:', (hpreact_fast - hpreact).abs().max())

max diff: tensor(4.7684e-07, grad_fn=<MaxBackward1>)


In [15]:
# backward pass
dhprebn = bngain*bnvar_inv/n * (n*dhpreact - dhpreact.sum(0) - n/(n-1)*bnraw*(dhpreact*bnraw).sum(0))
cmp('hprebn', dhprebn, hprebn)

hprebn          | exact: False | maxdiff: 9.313225746154785e-10
